In [1]:
import os

# 获取当前工作目录
current_dir = os.getcwd()
print("当前工作目录：", current_dir)

# 切换到上一层目录
parent_dir = os.path.dirname(current_dir)
os.chdir(parent_dir)
print("切换后的目录：", parent_dir)

当前工作目录： c:\Users\zhaoxs3\Documents\GitHub\NEMESIS\unit_test
切换后的目录： c:\Users\zhaoxs3\Documents\GitHub\NEMESIS


In [2]:
import QuantLib as ql
import pandas as pd

from devlib.market.curves.fdr import Fr007
from devlib.products.rates.irs.general_irs import *

In [3]:
from nemesis.products.rates import *
from nemesis.market.indices.interest_rate_index import InterestRateIndex, DataFrameFixingSource

####################################################################
#  NEMESIS ALPHA Version 0.1.0 - This build: 24 Jan 2025 at 10:42 #
####################################################################



In [12]:
today = ql.Date(25, 3, 2024)
ql.Settings.instance().evaluationDate = today
mkt_file_path = './unit_test/data/fr007_curve_data_20240325.xlsx'
deposit_mkt_data = pd.read_excel(mkt_file_path, sheet_name='deposit')
swap_mkt_data = pd.read_excel(mkt_file_path, sheet_name='swap')
fixing_data = pd.read_excel(mkt_file_path, sheet_name='fixing')

# swap_mkt_data = pd.DataFrame({
#     "Tenor": ["1M", "3M", "6M", "9M", "1Y", "2Y", "3Y", "4Y", "5Y", "7Y", "10Y"],
#     "Rate":  [0.0211375, 0.02086211, 0.02060056, 0.02013368, 0.01999084,
#               0.02001416, 0.02048767, 0.02112384, 0.02179811, 0.02269931, 0.02347403],
# })

fr007_curve = Fr007(
    today, deposit_mkt_data=deposit_mkt_data,
    swap_mkt_data=swap_mkt_data, fixing_data=fixing_data)

In [5]:
fixing_df = fixing_data.set_index('Date')
fixing_source = DataFrameFixingSource(
    fixing_df/100,
    fixing_col='Fixing',
    fallback_to_last=True,
 )


In [6]:
value_dt = Date(25,3,2024)

cal_type = CalendarTypes.CHINA_IB
cal = Calendar(cal_type)

index = InterestRateIndex(
    cal_type=cal_type,
    fixing_lag=1,
    spot_lag=1,
    tenor="1W",
)

settle_dt = cal.add_business_days(value_dt, 1)

notional = 1000000

fixed_type = SwapTypes.PAY
fixed_freq = FrequencyTypes.QUARTERLY
fixed_dc_type = DayCountTypes.ACT_365F

float_freq = FrequencyTypes.QUARTERLY
float_dc_type = DayCountTypes.ACT_365F

bd_type = BusDayAdjustTypes.MODIFIED_FOLLOWING
dg_type = DateGenRuleTypes.FORWARD
spread = 0

In [7]:
deposits = [
    InterestRateDeposit(
        effective_dt=settle_dt,
        maturity_dt_or_tenor="1W", # deposit_mkt_data["Tenor"].iloc[0],
        deposit_rate=deposit_mkt_data["Rate"].iloc[0],
        dc_type=DayCountTypes.ACT_365F,
        cal_type=cal_type,
        bd_type=BusDayAdjustTypes.FOLLOWING,
    )
]


In [16]:
# float_convention = ResetCompoundedFloatRateConvention(
#     multiplier=1.0,
#     spread=0.0,
#     compounding_type=CompoundingTypes.EXCLUDE_SPREAD,
#     reset_freq_type=FrequencyTypes.WEEKLY,
#     reset_bd_type=BusDayAdjustTypes.NONE,
#     reset_dg_type=DateGenRuleTypes.FORWARD_OVERSHOOT,
#  )
float_convention = FloatRateConvention(
    multiplier=1.0,
    spread=0.0,
)

swaps = [
    InterestRateSwap(
        effective_dt=settle_dt,
        term_dt_or_tenor="1M",
        fixed_leg_type=fixed_type,
        fixed_cpn=swap_mkt_data["Rate"].iloc[0],
        fixed_freq_type=FrequencyTypes.QUARTERLY,
        fixed_dc_type=fixed_dc_type,
        float_freq_type=FrequencyTypes.QUARTERLY,
        float_dc_type=float_dc_type,
        rate_index=index,
        float_convention=float_convention,
        notional=notional,
        payment_lag=0,
        cal_type=cal_type,
        bd_type=bd_type,
        dg_type=dg_type,
    )
] + [
    InterestRateSwap(
        effective_dt=settle_dt,
        term_dt_or_tenor=tenor,
        fixed_leg_type=fixed_type,
        fixed_cpn=rate,
        fixed_freq_type=fixed_freq,
        fixed_dc_type=fixed_dc_type,
        float_freq_type=float_freq,
        float_dc_type=float_dc_type,
        rate_index=index,
        float_convention=float_convention,
        notional=notional,
        payment_lag=0,
        cal_type=cal_type,
        bd_type=bd_type,
        dg_type=dg_type,
    )
    for tenor, rate in zip(swap_mkt_data["Tenor"][1:], swap_mkt_data["Rate"][1:])
]

In [17]:
curve = InterestRateCurve(
    value_dt,
    deposits,
    [],
    swaps,
    interp_type=InterpTypes.LINEAR_ZERO_RATES,
    is_index=True,
    currency="CNY",
 )

In [10]:
curve.print_table(curve.display_dts[1:])

,Date,ZR,DF
0,2024-04-02,2.39945,0.999474
1,2024-04-26,2.12511,0.998139
2,2024-06-26,2.08420,0.994704
3,2024-09-26,2.05647,0.989631
4,2024-12-26,2.00920,0.984922
5,2025-03-26,1.99471,0.980197
6,2026-03-26,1.99675,0.960799
7,2027-03-26,2.04482,0.940446
8,2028-03-27,2.10999,0.918904
9,2029-03-26,2.17964,0.896640


In [18]:
curve.print_table(curve.display_dts[1:])

,Date,ZR,DF
0,2024-04-02,2.39945,0.999474
1,2024-04-26,2.12084,0.998142
2,2024-06-26,2.08421,0.994704
3,2024-09-26,2.05647,0.989631
4,2024-12-26,2.00906,0.984923
5,2025-03-26,1.99456,0.980198
6,2026-03-26,1.99668,0.960801
7,2027-03-26,2.04479,0.940447
8,2028-03-27,2.10997,0.918905
9,2029-03-26,2.17962,0.896640
